# yhfiance에서 Brent 가격 뽑아오기

1. Brent 가격 (BZ=f)

- 국제 기준 원유 가격 그 자체임: 브랜트유 선물 종가를 기준으로 한 국제 유가
- 대부분의 유가 뉴스나 리포트에서 '국제유가 상승'이라 함은 얘 말함

2. WTI & Brent–WTI 스프레드

- '지역 간 가격차'로 수급 구조를 반영하는 보조 피처
- WTI는 미국 원유 선물인데, 미국 내 생산/재고/수출입 이슈에 더 민감함
- 스프레드 = brent 종가 - wti 종가

스프레드가 늘어나면 그니까 브렌트가 더 비싸면 -> 유럽/아시아 수급이 타이트

스프테드가 줄면 -> 미국산 원유 공급 증가 or 글로벌 수요 약화


**브렌트 자체가 유가의 절대수준**

**스프레드는 지역 간 수급 불균형 지표**

라서 경제적 맥락 읽힐라면 필요하다

In [11]:
import yfinance as yf
import pandas as pd

def make_brent_wti_features(start="2014-01-01", end=None):
    brent = yf.download("BZ=F", start=start, end=end, auto_adjust=False, progress=False)
    wti   = yf.download("CL=F", start=start, end=end, auto_adjust=False, progress=False)

    brent = brent.rename(columns=str.lower)
    wti   = wti.rename(columns=str.lower)

    df = pd.DataFrame(index=brent.index)
    df["brent_close"] = brent["close"]
    df["wti_close"] = wti["close"].reindex(df.index)

    # 스프레드
    df["brent_wti_spread"] = df["brent_close"] - df["wti_close"]

    # 수익률
    df["brent_ret_1d"] = df["brent_close"].pct_change(1)
    df["brent_ret_5d"] = df["brent_close"].pct_change(5)
    df["brent_ret_20d"] = df["brent_close"].pct_change(20)

    # 이동평균
    df["brent_ma_5"] = df["brent_close"].rolling(5).mean()
    df["brent_ma_20"] = df["brent_close"].rolling(20).mean()
    df["brent_ma_60"] = df["brent_close"].rolling(60).mean()

    # 변동성 proxy
    df["brent_vol_5d"] = df["brent_close"].pct_change().rolling(5).std()
    df["high_low_range"] = (brent["high"] - brent["low"]) / brent["close"]

    return df.dropna()

features = make_brent_wti_features()
print(features.tail())


            brent_close  wti_close  brent_wti_spread  brent_ret_1d  \
Date                                                                 
2025-11-03    64.889999  61.049999          3.840000     -0.002766   
2025-11-04    64.440002  60.560001          3.880001     -0.006935   
2025-11-05    63.520000  59.599998          3.920002     -0.014277   
2025-11-06    63.380001  59.430000          3.950001     -0.002204   
2025-11-07    63.630001  59.750000          3.880001      0.003944   

            brent_ret_5d  brent_ret_20d  brent_ma_5  brent_ma_20  brent_ma_60  \
Date                                                                            
2025-11-03     -0.011125      -0.008859   64.856000      63.8185    66.014333   
2025-11-04      0.000621      -0.015432   64.864000      63.7680    65.977833   
2025-11-05     -0.021565      -0.041208   64.584000      63.6315    65.934500   
2025-11-06     -0.024923      -0.028212   64.260001      63.5395    65.897000   
2025-11-07     -0.02213

In [12]:
features

,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range
Date,,,,,,,,,,,
2014-04-01,105.620003,99.739998,5.880005,-0.019859,-0.012805,-0.033669,107.262001,107.5065,107.855500,0.010387,0.025185
2014-04-02,104.790001,99.620003,5.169998,-0.007858,-0.020929,-0.027561,106.814001,107.3580,107.805667,0.010464,0.017273
2014-04-03,106.150002,100.290001,5.860001,0.012978,-0.015580,-0.018039,106.478001,107.2605,107.793333,0.012149,0.017146
2014-04-04,106.720001,101.139999,5.580002,0.005370,-0.012492,-0.020917,106.208002,107.1465,107.793167,0.012567,0.008527
2014-04-07,105.820000,100.440002,5.379997,-0.008433,-0.018003,-0.020910,105.820001,107.0335,107.767667,0.012856,0.013986
...,...,...,...,...,...,...,...,...,...,...,...
2025-11-03,64.889999,61.049999,3.840000,-0.002766,-0.011125,-0.008859,64.856000,63.8185,66.014333,0.009964,0.015102
2025-11-04,64.440002,60.560001,3.880001,-0.006935,0.000621,-0.015432,64.864000,63.7680,65.977833,0.005557,0.015673
2025-11-05,63.520000,59.599998,3.920002,-0.014277,-0.021565,-0.041208,64.584000,63.6315,65.934500,0.006487,0.023772


# EIA 데이터에서 수급 & 재고 관련한거 가져오기


### 상업 원유 재고

In [ ]:
m9fePZTks6kdpVDleRYzWYJMdZOXubKwwTHYuiPJ

In [ ]:
import requests
import pandas as pd

API_KEY = "여기에_본인_API_KEY"

url = "https://api.eia.gov/v2/petroleum/stoc/wstk/data/"

params = {
    "api_key": API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    "facets[process][]": "SAX",   # Ending Stocks Excluding SPR
    "start": "2014-01-01",
    "end": "2025-10-31",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000,
}

res = requests.get(url, params=params)
res.raise_for_status()
j = res.json()

print("total on server:", j["response"]["total"])
print("rows received  :", len(j["response"]["data"]))


In [51]:
import requests
import pandas as pd

API_KEY = "m9fePZTks6kdpVDleRYzWYJMdZOXubKwwTHYuiPJ"

url = "https://api.eia.gov/v2/petroleum/stoc/wstk/data/"

params = {
    "api_key": API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    "facets[process][]": "SAX",   
    "start": "2014-01-01",
    "end": "2025-10-31",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000,
}

res = requests.get(url, params=params)
res.raise_for_status()
j = res.json()

print("total on server:", j["response"]["total"])
print("rows received  :", len(j["response"]["data"]))


total on server: 4944
rows received  : 4944


In [52]:
df = pd.DataFrame(j["response"]["data"])
df.tail()

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
4939,2025-10-31,NUS,U.S.,EPC0,Crude Oil,SAX,Ending Stocks Excluding SPR,WCESTUS1,U.S. Ending Stocks excluding SPR of Crude Oil ...,421168,MBBL
4940,2025-10-31,R10,PADD 1,EPC0,Crude Oil,SAX,Ending Stocks Excluding SPR,WCESTP11,East Coast (PADD 1) Ending Stocks excluding SP...,8119,MBBL
4941,2025-10-31,R30,PADD 3,EPC0,Crude Oil,SAX,Ending Stocks Excluding SPR,WCESTP31,Gulf Coast (PADD 3) Ending Stocks excluding SP...,237478,MBBL
4942,2025-10-31,YCUOK,NA,EPC0,Crude Oil,SAX,Ending Stocks Excluding SPR,W_EPC0_SAX_YCUOK_MBBL,"Cushing, OK Ending Stocks excluding SPR of Cru...",22865,MBBL
4943,2025-10-31,R40,PADD 4,EPC0,Crude Oil,SAX,Ending Stocks Excluding SPR,WCESTP41,Rocky Mountain (PADD 4) Ending Stocks excludin...,23627,MBBL


In [44]:
df['area-name'].unique()

array(['NA', 'PADD 5', 'PADD 1', 'U.S.', 'PADD 4', 'PADD 3', 'PADD 2'],
      dtype=object)

In [45]:
df['process-name'].unique()

# Ending Stocks Excluding SPR : 상업용 원유 재고

array(['Ending Stocks Excluding SPR'], dtype=object)

In [39]:
# 공통 전처리: 날짜를 index로
df["date"] = pd.to_datetime(df["period"])
df = df.sort_values("date")

# 미국 전체만 사용 
us = df[df["area-name"] == "U.S."].copy()
us = us.sort_values("date")

# 1) 상업 원유 재고: Crude Oil + Ending Stocks Excluding SPR
crude = us[
    (us["product-name"].str.contains("Crude Oil", case=False)) &
    (us["process-name"] == "Ending Stocks Excluding SPR")
].copy()

crude = (
    crude[["date", "value"]]
    .drop_duplicates(subset="date", keep="last")
    .set_index("date")
    .sort_index()
    .rename(columns={"value": "crude_stock_level"})
)

# 2) Motor Gasoline 재고: Motor Gasoline + Ending Stocks
gas = us[
    (us["product-name"].str.contains("Motor Gasoline", case=False)) &
    (us["process-name"] == "Ending Stocks")
].copy()

gas = (
    gas[["date", "value"]]
    .drop_duplicates(subset="date", keep="last")
    .set_index("date")
    .sort_index()
    .rename(columns={"value": "gas_stock_level"})
)

# 3) Distillate Fuel Oil 재고: Distillate + Ending Stocks
dist = us[
    (us["product-name"].str.contains("Distillate", case=False)) &
    (us["process-name"] == "Ending Stocks")
].copy()

dist = (
    dist[["date", "value"]]
    .drop_duplicates(subset="date", keep="last")
    .set_index("date")
    .sort_index()
    .rename(columns={"value": "dist_stock_level"})
)

# 하나로 합치기 (날짜 기준 outer/inner 선택 가능, 기본은 inner)
stocks = pd.concat([crude, gas, dist], axis=1)

print(stocks.tail())


           crude_stock_level gas_stock_level dist_stock_level
date                                                         
2025-10-03            420261             NaN              NaN
2025-10-10           1288820             NaN              NaN
2025-10-17            422824             NaN              NaN
2025-10-24           1268745             NaN              NaN
2025-10-31            421168             NaN              NaN


### 파생변수들

In [41]:
# 숫자형으로 변환 (문자 → float)
for col in ["crude_stock_level", "gas_stock_level", "dist_stock_level"]:
    stocks[col] = pd.to_numeric(stocks[col], errors="coerce")

# WoW 변화량
stocks["crude_stock_wow_change"] = stocks["crude_stock_level"].diff()

# 5년 평균 대비
rolling_5yr = stocks["crude_stock_level"].rolling(window=5*52, min_periods=1).mean()
stocks["crude_stock_vs_5yr_avg"] = stocks["crude_stock_level"] / rolling_5yr -1

# 휘발유/증류유도 동일
stocks["gas_stock_vs_5yr"] = stocks["gas_stock_level"] / stocks["gas_stock_level"].rolling(260, min_periods=52).mean() - 1
stocks["dist_stock_vs_5yr"] = stocks["dist_stock_level"] / stocks["dist_stock_level"].rolling(260, min_periods=52).mean() - 1

In [43]:
stocks.tail()

,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr,dist_stock_vs_5yr
date,,,,,,,
2025-10-03,420261,NaN,NaN,3715.0,-0.484298,NaN,NaN
2025-10-10,1288820,NaN,NaN,868559.0,0.582304,NaN,NaN
2025-10-17,422824,NaN,NaN,-865996.0,-0.478508,NaN,NaN
2025-10-24,1268745,NaN,NaN,845921.0,0.565614,NaN,NaN
2025-10-31,421168,NaN,NaN,-847577.0,-0.480118,NaN,NaN
